# Uzbekistan Airways - Web Scraping Project

**Student:** Kamilla Tadjibaeva  
**Student ID:** 488381    
**Website:** https://www.uzairways.com/

In this project I scrape data from the Uzbekistan Airways website using different web scraping tools. This is a national airline from Uzbekistan, the country where I am from, that is why this project is not only useful but also personal for me :) The website has information about the airline's fleet, flight schedules, representative offices in different countries and news.

I will use:
- Requests and BeautifulSoup for static HTML content
- Selenium for pages with dynamic/interactive elements
- Scrapy for multi-page scraping
- regex for extracting data from text

## Imports

First importing all the libraries I will need in this project

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import json
import time
import os
import subprocess

In [2]:
os.makedirs("data", exist_ok=True)

headers = {
    "User-Agent": "student project"
}

## Fleet Information

**Tools used:** requests, BeautifulSoup, regex

The fleet page (https://www.uzairways.com/en/our-fleet) lists all the aircraft types operated by Uzbekistan Airways with their specifications.

I noticed that the h3 headings on the page contain embedded img tags whose alt text gets mixed into the heading text when using get_text(). I use BeautifulSoup's decompose() to remove the images before reading the name.

I also noticed that different aircraft have their specs in different formats on the website:
- Boeing models use format like: Length - 56,7 m
- Airbus models use format like: 44,5 m Length
- Pilatus uses: Length – 8 m

So I will use different regex patterns to handle both formats.


After inspecting the page I see the following: each aicraft type is under an h3 tag, so I will be using that for guidance and the fleet data itself is under div with class 'inner-content-text-wrapper'

In [3]:
url = "https://www.uzairways.com/en/our-fleet"
html = requests.get(url, headers=headers)
print("Status:", html.status_code)

bs = BeautifulSoup(html.text, "html.parser")

# the fleet data is inside a div with class 'inner-content-text-wrapper'
wrapper = bs.find("div", class_="inner-content-text-wrapper")

# each h3 tag is an aircraft type
h3_tags = wrapper.find_all("h3")
print(f"found {len(h3_tags)} aircraft sections")


Status: 200
found 7 aircraft sections


I also noticed that h3 tags which as described above have the aircrafts - they have image embedded into the titles, which is not a common practice and likely is a mistake of developer, because it is also not consistent across all the aircrafts. Therefore I need to remove the img inside with its alt text. 
Below I am also getting all the specs of airplanes and using specific regex for specific aircraft types since they differ.

In [4]:
results = []

for h3 in h3_tags:
    #some of the h3 titles have image tags inside, which is very inconsitent across this site
    # removing <img> tags inside h3 to avoid alt text being mixed into the name
    for img in h3.find_all("img"):
        img.decompose()
    aircraft_name = h3.get_text(strip=True)
    
    if not aircraft_name:
        continue
    
    ul = h3.find_next("ul")
    if not ul:
        continue
    
    # getting all airplanes spec text lines
    specs = [li.get_text(strip=True) for li in ul.find_all("li")]
    
    length = None
    wingspan = None
    speed = None
    range_km = None
    capacity = None
    economy = None
    business = None
    
    for spec in specs:
        # I need to handle two formats:
        # FORMAT 1: "Length - 56,7 m"  (Boeing, Pilatus)
        # FORMAT 2: "44,5 m Length"    (Airbus)
        
        # Length
        m = re.search(r'Length\s*[-–]\s*([\d,\.]+)\s*m', spec)
        if m:
            length = m.group(1).replace(",", ".")
        m = re.search(r'([\d,\.]+)\s*m\s+Length', spec)
        if m:
            length = m.group(1).replace(",", ".")
        
        # Wingspan
        m = re.search(r'Wingspan\s*[-–]\s*([\d,\.\s–-]+)\s*m', spec)
        if m:
            wingspan = m.group(1).replace(",", ".").strip()
        m = re.search(r'([\d,\.\s–-]+)\s*m\s+Wingspan', spec)
        if m:
            wingspan = m.group(1).replace(",", ".").strip()
        
        # Cruising speed
        m = re.search(r'(\d+)\s*km/h', spec)
        if m:
            speed = m.group(1)
        
        # Range 
        m = re.search(r'Range\s*[-–]\s*(?:from\s*)?([\d,\.]+)', spec)
        if m:
            range_km = m.group(1).replace(",", "")
        m = re.search(r'([\d,\.]+)\s*(?:thousands of\s*)?km\s+Range', spec)
        if m:
            val = m.group(1).replace(",", ".")
            if "thousands" in spec:
                range_km = str(int(float(val) * 1000))
            else:
                range_km = val
        
        # Passenger capacity
        m = re.search(r'(Passenger capacity|Number of seats)\s*[-–]\s*([\d-]+)', spec)
        if m:
            capacity = m.group(2)
        m = re.search(r'(\d+)\s+Passenger capacity', spec)
        if m:
            capacity = m.group(1)
        
        # Economy seats
        m = re.search(r'(\d+)\s+[Ee]conomy', spec)
        if m:
            economy = m.group(1)
        
        # Business seats
        m = re.search(r'(\d+)\s+[Bb]usiness', spec)
        if m:
            business = m.group(1)
    
    results.append({
        "Aircraft": aircraft_name,
        "Economy Seats": economy,
        "Business Seats": business,
        "Length (m)": length,
        "Wingspan (m)": wingspan,
        "Cruising Speed (km/h)": speed,
        "Range (km)": range_km,
        "Passenger Capacity": capacity,
    })
    print(f"  Extracted: {aircraft_name}")

print(f"\nTotal aircraft types: {len(results)}")

print(f"Example Aircraft: {results[0]["Aircraft"]}, {results[0]["Business Seats"]}, {results[0]["Passenger Capacity"]}")


  Extracted: Boeing 787-8
  Extracted: Boeing 787-8
  Extracted: Boeing 767-300ER
  Extracted: A320-214
  Extracted: Airbus A 321-200NX
  Extracted: Airbus A 330-200
  Extracted: Pilatus PC-24

Total aircraft types: 7
Example Aircraft: Boeing 787-8, 24, 270


Finilizing the results and putting them into a dataframe and later into csv file so that it can be used later by someone (if I was to share it with someone)

In [5]:
df_fleet = pd.DataFrame(results)
df_fleet.to_csv("data/fleet.csv", index=False)
df_fleet

,Aircraft,Economy Seats,Business Seats,Length (m),Wingspan (m),Cruising Speed (km/h),Range (km),Passenger Capacity
0,Boeing 787-8,246,24,56.7,60.1,930,12700,270
1,Boeing 787-8,222,24,56.7,60.1,930,12700,246
2,Boeing 767-300ER,232,15,54.9,47.6,800,9000,247
3,A320-214,138,12,37.6,34.1,850,3800,150
4,Airbus A 321-200NX,172,16,44.5,34.1 – 35.8,840,7400,188
5,Airbus A 330-200,248,18,58.37,60.3,930,12000,266
6,Pilatus PC-24,NaN,NaN,8,17,815,3300,6-8


## Part 2: Flight Schedule

**Tools used:** requests, BeautifulSoup, Selenium

The schedule page (https://www.uzairways.com/en/schedule) shows current departure and arrival information for flights. The page has 4 tab combinations:
- International flights / Local flights
- Departure / Arrival

First I use requests + BeautifulSoup to explore the raw HTML and discover how many tables embedded in the page. Then I use Selenium to click through each tab combination and extract the data the way a real user would interact with the page.


In [6]:

url = "https://www.uzairways.com/en/schedule"
html = requests.get(url, headers=headers)

bs_explore = BeautifulSoup(html.text, "html.parser")
all_tables = bs_explore.find_all("table")
print(f"\nfound {len(all_tables)} <table> elements in the HTML")

# checking parent divs to see their IDs and CSS classes
print("\ndiv containers for each table:")
for table in all_tables:
    parent_div = table.find_parent("div", id=True)
    if parent_div:
        div_id = parent_div.get("id", "no id")
        div_class = parent_div.get("class", [])
        print(f"  div id='{div_id}'  classes={div_class}")

# all 4 tables are already in the HTML source — JavaScript just toggles their visibility
# now I will use Selenium to click each tab and extract data as a real user would



found 4 <table> elements in the HTML

div containers for each table:
  div id='inter_depart'  classes=['responsive-table-box']
  div id='inter_arrival'  classes=['responsive-table-box']
  div id='region_depart'  classes=['responsive-table-box']
  div id='region_arrival'  classes=['responsive-table-box']


Now I use Selenium to actually click through the tabs and scrape the visible tables. The site has a cookie banner that needs to be closed first, and normal .click() does not work on the tab buttons here so I use execute_script instead.

In [7]:

from selenium import webdriver 
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

path = ChromeDriverManager().install()
service = Service(executable_path=path)
options = webdriver.ChromeOptions()
# -- headless to not have a chrome pop up
# options.add_argument("--headless")
driver = webdriver.Chrome(service=service, options=options)

url = "https://www.uzairways.com/en/schedule"
driver.get(url)
driver.implicitly_wait(10)

# closeing the cookie banner first
# the banner has a close button with class 'notifyCookies__button'
try:
    cookie_btn = driver.find_element(By.CSS_SELECTOR, "button.notifyCookies__button")
    cookie_btn.click()
    time.sleep(1)
    print("Cookie banner closed")
except Exception:
    print("No cookie banner found")

results = []

# the page has two flight-type tabs (International / Local)
# each has its own Departure / Arrival radio buttons inside:
# .switchFlight__item-international -> inter_depart, inter_arrival
# .switchFlight__item-regional -> region_depart, region_arrival

tab_combinations = [
    {
        "label": "International Departures",
        "flight_tab": "span.inter-flights",
        # since class name input-radio is the same for departures and arrivals I take the 1st one nth-of-type(1) as departures, and 2nd as arrivals , as I inspected in the inspector in browser
        "dir_tab": ".switchFlight__item-international label.input-radio:nth-of-type(1)",
        "table_id": "inter_depart",
    },
    {
        "label": "International Arrivals",
        "flight_tab": "span.inter-flights",
        #Since both labels have the same class nth-of-type(2) is to choose 2nd one
        "dir_tab": ".switchFlight__item-international label.input-radio:nth-of-type(2)",
        "table_id": "inter_arrival",
    },
    {
        "label": "Local Departures",
        "flight_tab": "span.input-toggler-span.region-flights",
        "dir_tab": ".switchFlight__item-regional label.input-radio:nth-of-type(1)",
        "table_id": "region_depart",
    },
    {
        "label": "Local Arrivals",
        "flight_tab": "span.input-toggler-span.region-flights",
        "dir_tab": ".switchFlight__item-regional label.input-radio:nth-of-type(2)",
        "table_id": "region_arrival",
    },
]

# column names matching the 9 columns in each flight table
col_names = ["Flight", "City", "Date", "Scheduled", "Estimated",
             "Aircraft", "Fare Classes", "Gate", "Status"]

for config in tab_combinations:
    try:
        flight_btn = driver.find_element(By.CSS_SELECTOR, config["flight_tab"])
        # does not work with click on this site with flight_btn.click(), so I am using execute_script instead
        driver.execute_script("arguments[0].click();", flight_btn)
        time.sleep(1)

        dir_btn = driver.find_element(By.CSS_SELECTOR, config["dir_tab"])
        driver.execute_script("arguments[0].click();", dir_btn)
        time.sleep(1)

        table = driver.find_element(By.CSS_SELECTOR, f"#{config['table_id']} table")
        rows = table.find_elements(By.TAG_NAME, "tr")[1:]  # skip header row

        for row in rows:
            cells = row.find_elements(By.TAG_NAME, "td") # td = table data, so cell in a row
            if not cells:
                continue  # skip completely empty rows

            row_data = {"Type": config["label"]}
            for i, col in enumerate(col_names):
                row_data[col] = cells[i].text if i < len(cells) else ""
            results.append(row_data)

        print(f"{config['label']}: {len(rows)} flights")

    except Exception as e:
        print(f"Error for {config['label']}: {e}")

driver.quit()


Cookie banner closed
International Departures: 59 flights
International Arrivals: 57 flights
Local Departures: 14 flights
Local Arrivals: 16 flights


Saving the flights into a dataframe and csv

In [8]:

df_schedule = pd.DataFrame(results)
df_schedule.to_csv("data/flights.csv", index=False)

df_schedule.head(30)


,Type,Flight,City,Date,Scheduled,Estimated,Aircraft,Fare Classes,Gate,Status
0,International Departures,427,MUMBAI,11.05.2026,23:20,23:20,32N,"Y39,Y40",B6,Delayed
1,International Departures,505,BEIJING,11.05.2026,23:30,23:30,789,"Y32,Y33",B2,Delayed
2,International Departures,281,ISTANBUL (Istanbul International Airport),12.05.2026,01:10,01:10,320,"Y29,Y30,Y31",B9,On schedule
3,International Departures,421,DELHI,12.05.2026,01:25,01:25,763,"Y35,Y36,Y37",B11,On schedule
4,International Departures,465,ISLAMABAD,12.05.2026,03:40,03:40,32N,,B9,On schedule
5,International Departures,9613,MOSCOW (Vnukovo),12.05.2026,05:40,05:40,32A,,B8,On schedule
6,International Departures,611,MOSCOW (Vnukovo),12.05.2026,06:30,06:30,32Q,,B9,On schedule
7,International Departures,101,NEW-YORK (JFK International Airport),12.05.2026,06:45,06:45,787,,B5,On schedule
8,International Departures,685,SOCHI,12.05.2026,07:05,07:05,32N,,B8,On schedule
9,International Departures,9657,UFA,12.05.2026,07:30,07:30,32A,,,On schedule


## Representative Offices

**Tools used:** Selenium, regex

The offices page (https://www.uzairways.com/en/representative-offices) lists all Uzbekistan Airways offices worldwide. Each country is hidden inside a collapsible "spoilerbox" - I need to click on the country name to expand it and see the details.

I need Selenium here because clicking is required. After expanding each country I use regex to get phone numbers, fax numbers, emails and addresses from the raw text. Some offices have info in Russian too (yes we speak both Uzbek and Russian in my country, but no idea why they kept it in Russian on English version of the site) (like Факс instead of Fax) so the regex needs to handle both.

In [9]:
service = Service(executable_path=path)
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)

url = "https://www.uzairways.com/en/representative-offices"
driver.get(url)
driver.implicitly_wait(10)

# closeing the cookie banner first
try:
    cookie_btn = driver.find_element(By.CSS_SELECTOR, "button.notifyCookies__button")
    cookie_btn.click()
    time.sleep(1)
    print("Cookie banner closed")
except Exception:
    print("No cookie banner found")

# each country box is in a "spoilerbox" div
spoilerboxes = driver.find_elements(By.CSS_SELECTOR, "div.spoilerbox")
print(f"Found {len(spoilerboxes)} countries")

results = []

for sb in spoilerboxes:
    try:
        heading = sb.find_element(By.CSS_SELECTOR, "div.heading")
        country = heading.text.strip()
        
        # click to expand the content (if not already open)
        #active is present in some countries class names like Uzbekistan , which means they are already expanded and no need to click
        if "active" not in sb.get_attribute("class"):
            driver.execute_script("arguments[0].click();", heading)
            #waiting to load conent after click
            time.sleep(1)
        
        content = sb.find_element(By.CSS_SELECTOR, "div.content")
        details = content.text.strip()
        
        # city names are in h3 tags inside the content
        cities = content.find_elements(By.TAG_NAME, "h3")
        city_names = [c.text.strip() for c in cities if c.text.strip()]
        
        # use regex to extract phone numbers
        phones = re.findall(r'(?:Phone|Tel)[.:\s]*([\+\d\s\-\(\);/]+)', details, re.IGNORECASE)
        phones = [p.strip().rstrip(';').strip() for p in phones if len(p.strip()) > 4]
        
        # use regex to extract fax numbers
        faxes = re.findall(r'(?:Fax|Факс)[.:\s]*([\+\d\s\-\(\);/]+)', details, re.IGNORECASE)
        faxes = [f.strip().rstrip(';').strip() for f in faxes if len(f.strip()) > 4]
        
        # use regex to extract emails
        emails = re.findall(r'[\w\.-]+@[\w\.-]+\.\w+', details)
        
        # use regex to extract addresses (text after "Address:")
        addresses = re.findall(r'Address[:\s]+(.+)', details)
        addresses = [a.strip() for a in addresses if a.strip()]
        
        results.append({
            "Country": country,
            "Cities": ", ".join(city_names) if city_names else "",
            "Addresses": "; ".join(addresses) if addresses else "",
            "Phones": "; ".join(phones) if phones else "",
            "Faxes": "; ".join(faxes) if faxes else "",
            "Emails": "; ".join(emails) if emails else "",
            "Full Info": details,
        })
        print(f"  {country} - {len(city_names)} cities")
    
    except Exception as e:
        print(f"  Error: {e}")

driver.quit()

Cookie banner closed
Found 27 countries
  Uzbekistan - 1 cities
  Azerbaijan - 1 cities
  Georgia - 1 cities
  Indonesia - 1 cities
  China - 4 cities
  Great Britain - 1 cities
  India - 2 cities
  Spain - 1 cities
  Italy - 2 cities
  Israel - 1 cities
  Japan - 1 cities
  Kazakhstan - 2 cities
  South Korea - 1 cities
  Kyrgyzstan - 1 cities
  Latvia - 1 cities
  Malaysia - 1 cities
  Russia - 5 cities
  Saudi Arabia - 1 cities
  Thailand - 1 cities
  Turkey - 1 cities
  United Arab Emirates - 1 cities
  USA - 1 cities
  Germany - 2 cities
  Belarus - 1 cities
  Tajikistan - 1 cities
  Pakistan - 1 cities
  France - 1 cities


Saving the offices data into a dataframe and csv, dropping the "Full Info" column from the display since it is too long

In [10]:
df_offices = pd.DataFrame(results)
df_offices.to_csv("data/offices.csv", index=False)
df_offices[["Country", "Cities", "Addresses", "Phones", "Faxes", "Emails"]]

,Country,Cities,Addresses,Phones,Faxes,Emails
0,Uzbekistan,Tashkent,"House 41, A.Temur Av., Tashkent,100060",(99878) 140-02-00,,
1,Azerbaijan,Baku,"H.Aslanov str.113, Baku",+994 12-493-01-40; +994 12-598-31-20,,bak@uzairways.com
2,Georgia,Tbilisi,,+995-322-195-800,,tbs@uzairways.com
3,Indonesia,Jakarta,,+62215205044,,
4,China,"Beijing, Urumqi, Shanghai, Tianjin","Jian Guo Men Wai 19 Citic Building 201 B, 1000...",(86-10) 65-00-64-42\n+86-10-8526-3212; (86) 99...,(86-10) 65-25-38-67; (86) 991 259 70 40; +86-2...,bjs@uzairways.com; urc@uzairways.com; sha@uzai...
5,Great Britain,London,,+44-020-7034-2090,+44-020-7371-1256,london@uzairways.co.uk; lon@uzairways.com
6,India,"Delhi, Mumbai","Rooms #108, 109, 110, 1-st floor, Narain Manzi...",+91 11 45058458; 022 4971 0208,,del@uzairways.com; sale.india@uzairways.com; s...
7,Spain,Madrid,Summerwind; Jorge Juan 68 1D,+34 910 785 597; 91 401 57 01,,Uzbekistanairways@swgsa.com; rom@uzairways.com...
8,Italy,"Milan, Rome","Via Marsala 36B, 21013, Gallarate (VA); Via Cl...",+39 03311086402; +39 03311086402,,rom@uzairways.com; rom@uzairways.com
9,Israel,Tel-Aviv,"Migdalor Building1, Ben Yehuda, Tel-Aviv, 63801",(972-3) 510-46-85,,tlv@uzairways.com; uzairways@bezeqint.net


## News Articles

**Tools used:** Scrapy, requests

The news section (https://www.uzairways.com/en/press-center/news) has a lot of articles spread across multiple pages, so I use Scrapy here since it is designed for exactly this - crawling through many pages automatically.

The spider code is in a separate file `news_scraper.py`. I run it as a subprocess because of the Twisted reactor issue we saw in class (Scrapy can only start once per Python session). The spider visits the first 5 pages of news, follows each article link, and extracts the title, date, body text and image URLs.

After scraping I also download the actual article images to a local folder using requests.

In [11]:
# the spider code is in news_scraper.py
# running it as a subprocess because Scrapy's Twisted reactor can only start once per Python session

result = subprocess.run(["python", "news_scraper.py"], capture_output=True, text=True)
result

CompletedProcess(args=['python', 'news_scraper.py'], returncode=0, stdout='', stderr='')

Now I load the scraped JSON, download the images from the URLs the spider collected, and put everything into a dataframe. I clean the article titles with regex to use them as image filenames since the original URLs had unreadable encoded characters.

In [12]:
with open("data/news_scrapy.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# download images to data/images/ folder
os.makedirs("data/images", exist_ok=True)

for article in data:
    local_paths = []
    # create a clean name from the article title for the image filename
    #re.sub(pattern, replacement, string)  :  finds all matches of a pattern in a string and replaces them with something else.
    clean_title = re.sub(r'[^a-zA-Z0-9]+', '_', article.get("title", "untitled")).strip('_').lower()[:50]
    ext = "jpg"
    
    for idx, img_url in enumerate(article.get("image_urls", []), start=1):
        filename = f"{clean_title}_{idx}.{ext}"
        filepath = os.path.join("data/images", filename)
        
        if not os.path.exists(filepath):
            try:
                img_data = requests.get(img_url, headers={"User-Agent": "student project"}).content
                with open(filepath, "wb") as img_file:
                    img_file.write(img_data)
            except Exception as e:
                print(f"  Error downloading {filename}: {e}")
                continue
        local_paths.append(filepath)
    
    article["image_urls_str"] = "; ".join(article.get("image_urls", []))
    article["local_images"] = "; ".join(local_paths) if local_paths else ""

df_news = pd.DataFrame(data)
df_news.to_csv("data/news.csv", index=False)
print(f"Scraped {len(df_news)} news articles")

df_news.head(10)

Scraped 45 news articles


,title,date,url,body,image_urls,image_urls_str,local_images
0,Official welcome ceremony held for Czech Prime...,30 April 2026,https://www.uzairways.com/en/press-center/news...,"On April 30, an official welcome ceremony was ...",[https://www.uzairways.com/sites/default/files...,https://www.uzairways.com/sites/default/files/...,data/images/official_welcome_ceremony_held_for...
1,Uzbek–Czech relations enter a new stage of dev...,30 April 2026,https://www.uzairways.com/en/press-center/news...,Negotiations were held between the President o...,[https://www.uzairways.com/sites/default/files...,https://www.uzairways.com/sites/default/files/...,data/images/uzbek_czech_relations_enter_a_new_...
2,Uzbekistan and the Czech Republic sign package...,30 April 2026,https://www.uzairways.com/en/press-center/news...,"Following talks held in Tashkent, President of...",[https://www.uzairways.com/sites/default/files...,https://www.uzairways.com/sites/default/files/...,data/images/uzbekistan_and_the_czech_republic_...
3,Meeting with Students of Turin Polytechnic Uni...,01 May 2026,https://www.uzairways.com/en/press-center/news...,The development of national aviation is imposs...,[https://www.uzairways.com/sites/default/files...,https://www.uzairways.com/sites/default/files/...,data/images/meeting_with_students_of_turin_pol...
4,Uzbekistan Airways Successfully Completes 10th...,01 May 2026,https://www.uzairways.com/en/press-center/news...,Uzbekistan Airways JSC has successfully passed...,[],,
5,New Horizons of Comfort: Pay for Uzbekistan Ai...,01 May 2026,https://www.uzairways.com/en/press-center/news...,"The national airline, Uzbekistan Airways, is c...",[],,
6,Uzbekistan Airways is moving to a new terminal...,04 May 2026,https://www.uzairways.com/en/press-center/news...,"Starting May 7, the airline’s flights will ope...",[],,
7,Honoring the Heroes: Uzbekistan Airways Paid T...,05 May 2026,https://www.uzairways.com/en/press-center/news...,On the eve of the Day of Remembrance and Honor...,[https://www.uzairways.com/sites/default/files...,https://www.uzairways.com/sites/default/files/...,data/images/honoring_the_heroes_uzbekistan_air...
8,A Historic Event in the Civil Aviation of the ...,08 May 2026,https://www.uzairways.com/en/press-center/news...,"Today, May 8, a truly historic event took plac...",[https://www.uzairways.com/sites/default/files...,https://www.uzairways.com/sites/default/files/...,data/images/a_historic_event_in_the_civil_avia...
9,The term of office of Shuhrat Shavkatovich Khu...,06 April 2026,https://www.uzairways.com/en/press-center/news...,By decision of the Supervisory Board and the G...,[],,


Expanding the column width to see the full content of each cell in the dataframe

In [13]:
pd.set_option("display.max_colwidth", None)  # show full cell content, not truncated
df_news.head(10)

,title,date,url,body,image_urls,image_urls_str,local_images
0,Official welcome ceremony held for Czech Prime Minister,30 April 2026,https://www.uzairways.com/en/press-center/news/official-welcome-ceremony-held-czech-prime-minister,"On April 30, an official welcome ceremony was held at the Kuksaroy Residence for the Prime Minister of the Czech Republic Andrej Babiš, who is on an official visit to Uzbekistan. A guard of honor was lined up in the square outside the residence. President Shavkat Mirziyoyev warmly welcomed the distinguished guest and invited him to the podium. The military band performed the national anthems of both countries. The leaders inspected the guard of honor and greeted members of the official delegations. Following the ceremony and a joint photo session, high-level negotiations began. Photo: Press Service of the President of the Republic of Uzbekistan\",[https://www.uzairways.com/sites/default/files/inline-images/2_16.jpg],https://www.uzairways.com/sites/default/files/inline-images/2_16.jpg,data/images/official_welcome_ceremony_held_for_czech_prime_min_1.jpg
1,Uzbek–Czech relations enter a new stage of development,30 April 2026,https://www.uzairways.com/en/press-center/news/uzbek-czech-relations-enter-new-stage-development,"Negotiations were held between the President of the Republic of Uzbekistan Shavkat Mirziyoyev, and the Prime Minister of the Czech Republic Andrej Babiš, in a narrow format and with the participation of the two countries’ delegations at the Kuksaroy Residence. The sides discussed the current state and prospects for further development of practical cooperation between Uzbekistan and the Czech Republic. At the beginning of the meeting, the President emphasized that the current visit would be a breakthrough and would open a qualitatively new stage in the history of multifaceted Uzbek–Czech relations. The dynamic development of cooperation across all areas was noted with satisfaction. Close contacts have been established at the level of governments, ministries, and agencies, while business and humanitarian exchanges have intensified. Friendship groups have been established in the parliaments, and the Intergovernmental Commission is operating effectively. Bilateral trade turnover has nearly doubled in recent years. There are currently 37 joint ventures with Czech capital successfully operating in Uzbekistan, and cooperation is expanding across a broad range of new areas. Particular attention was paid to increasing mutual trade. A target was set to raise trade turnover to $1 billion, including by expanding the range of traded goods. To stimulate trade, the first Uzbek certification branch is being established in the Czech Republic. In cooperation with Czech partners, a modern laboratory for the certification of Euro-6 standard vehicles and quantum measurement standards is also under construction. An agreement was reached to prepare a Technological Cooperation Program with leading Czech companies, including the implementation of projects in mechanical engineering, green energy, geology and critical raw materials, chemistry, pharmaceuticals, and other sectors. Prospects for cooperation in infrastructure development, the creation of smart cities, engineering, and digitalization were also noted. In this context, the project initiatives presented at the Uzbek–Czech Business Forum held the previous day were supported. The sides welcomed the intentions of the Export Credit Insurance Agency and the Export Bank of the Czech Republic to support project implementation in Uzbekistan. To advance the economic agenda, it was proposed to establish a Business Council and to hold the next meeting of the Intergovernmental Commission in Tashkent this August. In the cultural and humanitarian sphere, both sides confirmed their interest in expanding educational and academic exchanges, including through dual degree programs. The importance of regularly holding cultural and film days, exhibitions, and concerts, as we

In [14]:
pd.reset_option("display.max_colwidth")